# `eia(series, start, end=None)` and `eia_series(dataset=None, frequency=None)`

These functions give US energy data from `indicators.eia`, oldest first. The data comes from the U.S. Energy Information Administration (EIA). It has monthly spot prices for crude oil, refined products and natural gas, and monthly US crude oil production. It also has weekly data: petroleum stocks, refinery utilization and product supplied from the Weekly Petroleum Status Report, and natural gas in storage.

`eia_series()` gives the dataset, name, units, frequency and category of each series id. Each cell reads prod as `readonly_user`, and does not write. You can change the arguments and run a cell again.

Use the EIA series id (`RWTC`, `WCESTUS1`) or the name of a series, in any case and spacing. The functions do not give periods with no value, and do not fill them.

> **About the data shown.** Source: U.S. Energy Information Administration (Nov 2026), https://www.eia.gov/opendata/ (accessed November 26, 2026). EIA data is public domain. The values are the EIA values, in the EIA units. arora_data has no affiliation with EIA, and EIA does not endorse it. This notebook is not investment advice.

In [1]:
from datetime import date, timedelta

import polars as pl

from arora_data.dates import today_et
from arora_data.eia import EiaSeries, EiaValue
from arora_data.frames import to_df
from arora_data.sync import eia, eia_series


def side_by_side(ids, start):
    """The values of some series, one column for each series, joined on date."""
    frames = [to_df(eia(i, start), EiaValue).select("date", pl.col("value").alias(n)) for i, n in ids.items()]
    joined = frames[0]
    for f in frames[1:]:
        joined = joined.join(f, on="date", how="full", coalesce=True)
    return joined.sort("date")


def weekly(cell):
    """Run cell only when the table holds weekly series. The main repo's EIA-ADD sources (#312) write them."""
    if "weekly" not in set(series["frequency"]):
        return print("The table holds no weekly series yet.")
    return cell()

## The series
Each call reads the series from the table, thus a new series needs no code change. The labels come from the latest row of each series. `first_date` and `latest_date` are the first and latest periods that have a value.

In [2]:
series = to_df(eia_series(), EiaSeries)
print(series.height, "series")
series.select("series_id", "dataset", "name", "units", "frequency", "first_date", "latest_date")

7 series


series_id,dataset,name,units,frequency,first_date,latest_date
str,str,str,str,str,date,date
"""EER_EPD2F_PF4_Y35NY_DPG""","""eia/petroleum_spot""","""NY Harbor No.2 heating oil spo…","""$/GAL""","""monthly""",1986-06-01,2026-10-01
"""EER_EPLLPA_PF4_Y44MB_DPG""","""eia/petroleum_spot""","""Mont Belvieu propane spot""","""$/GAL""","""monthly""",1992-06-01,2026-10-01
"""EER_EPMRU_PF4_RGC_DPG""","""eia/petroleum_spot""","""Gulf Coast regular gasoline sp…","""$/GAL""","""monthly""",1986-06-01,2026-10-01
"""MCRFPUS2""","""eia/crude_production""","""US crude oil field production""","""MBBL/D""","""monthly""",1920-01-01,2026-08-01
"""RBRTE""","""eia/petroleum_spot""","""Brent crude spot (Europe)""","""$/BBL""","""monthly""",1987-05-01,2026-10-01
"""RNGWHHD""","""eia/natgas""","""Henry Hub natural gas spot""","""$/MMBtu""","""monthly""",1997-01-01,2026-10-01
"""RWTC""","""eia/petroleum_spot""","""WTI crude spot (Cushing)""","""$/BBL""","""monthly""",1986-01-01,2026-10-01


## One dataset or one frequency
`eia_series(dataset=...)` and `eia_series(frequency=...)` give only the series that have that value. You can use the two filters together. The number of series for each dataset and frequency:

In [3]:
series.group_by("dataset", "frequency").agg(pl.len().alias("series"), pl.col("first_date").min()).sort("frequency", "dataset")

dataset,frequency,series,first_date
str,str,u32,date
"""eia/crude_production""","""monthly""",1,1920-01-01
"""eia/natgas""","""monthly""",1,1997-01-01
"""eia/petroleum_spot""","""monthly""",5,1986-01-01


In [4]:
for s in eia_series(dataset="eia/petroleum_spot", frequency="monthly"):
    print(f"{s.series_id:<26} {s.units:<7} {s.name}")

EER_EPD2F_PF4_Y35NY_DPG    $/GAL   NY Harbor No.2 heating oil spot
EER_EPLLPA_PF4_Y44MB_DPG   $/GAL   Mont Belvieu propane spot
EER_EPMRU_PF4_RGC_DPG      $/GAL   Gulf Coast regular gasoline spot
RBRTE                      $/BBL   Brent crude spot (Europe)
RWTC                       $/BBL   WTI crude spot (Cushing)


## One series, by id or by name
The WTI crude spot price (Cushing) for the last 12 months, in dollars per barrel. The id and the name, in any case and spacing, give the same rows. A monthly value has the first day of its month.

In [5]:
year = today_et() - timedelta(days=366)
assert eia("RWTC", year) == eia("rwtc", year) == eia("  wti crude SPOT (cushing) ", year)
to_df(eia("RWTC", year), EiaValue)

series_id,name,date,value,units
str,str,date,"decimal[38,8]",str
"""RWTC""","""WTI crude spot (Cushing)""",2025-12-01,57.97000000,"""$/BBL"""
"""RWTC""","""WTI crude spot (Cushing)""",2026-01-01,60.04000000,"""$/BBL"""
"""RWTC""","""WTI crude spot (Cushing)""",2026-02-01,64.51000000,"""$/BBL"""
"""RWTC""","""WTI crude spot (Cushing)""",2026-03-01,91.38000000,"""$/BBL"""
"""RWTC""","""WTI crude spot (Cushing)""",2026-04-01,100.32000000,"""$/BBL"""
…,…,…,…,…
"""RWTC""","""WTI crude spot (Cushing)""",2026-06-01,84.81000000,"""$/BBL"""
"""RWTC""","""WTI crude spot (Cushing)""",2026-07-01,80.46000000,"""$/BBL"""
"""RWTC""","""WTI crude spot (Cushing)""",2026-08-01,83.90000000,"""$/BBL"""


## Brent and WTI
The two crude benchmarks for the last 12 months, in dollars per barrel, and the Brent price minus the WTI price. Brent is a North Sea price, not a US price.

In [6]:
side_by_side({"RBRTE": "Brent", "RWTC": "WTI"}, year).with_columns((pl.col("Brent") - pl.col("WTI")).alias("Brent - WTI"))

date,Brent,WTI,Brent - WTI
date,"decimal[38,8]","decimal[38,8]","decimal[38,8]"
2025-12-01,62.54000000,57.97000000,4.57000000
2026-01-01,66.60000000,60.04000000,6.56000000
2026-02-01,70.89000000,64.51000000,6.38000000
2026-03-01,103.13000000,91.38000000,11.75000000
2026-04-01,117.29000000,100.32000000,16.97000000
…,…,…,…
2026-06-01,85.40000000,84.81000000,0.59000000
2026-07-01,83.76000000,80.46000000,3.30000000
2026-08-01,91.08000000,83.90000000,7.18000000


## Units
The values are in the EIA units. "MBBL" is thousand barrels, "MBBL/D" is thousand barrels per day, and "BCF" is billion cubic feet. The units of the series:

In [7]:
series.group_by("units").agg(pl.col("series_id").sort().str.join(", ").alias("series")).sort("units")

units,series
str,str
"""$/BBL""","""RBRTE, RWTC"""
"""$/GAL""","""EER_EPD2F_PF4_Y35NY_DPG, EER_E…"
"""$/MMBtu""","""RNGWHHD"""
"""MBBL/D""","""MCRFPUS2"""


## Weekly petroleum stocks
US crude oil stocks without the Strategic Petroleum Reserve (SPR) and with it, for the last 8 weeks, in thousand barrels. A weekly value has the last day of its week, as EIA gives it. The SPR is the difference between the two series.

In [8]:
weeks = today_et() - timedelta(weeks=8)
weekly(lambda: side_by_side({"WCESTUS1": "without SPR", "WCRSTUS1": "with SPR"}, weeks) \
    .with_columns((pl.col("with SPR") - pl.col("without SPR")).alias("SPR"), pl.col("date").dt.strftime("%A").alias("day")))

The table holds no weekly series yet.


## Natural gas in storage
Working gas in storage for the Lower 48 and its 5 regions, for the last 4 weeks, in billion cubic feet. The last column is the Lower 48 value minus the sum of the 5 regions.

In [9]:
def gas():
    stored = eia_series(dataset="eia/weekly_gas_storage")
    regions = {s.series_id: (s.name or s.series_id).removeprefix("Working gas in storage, ") for s in stored}
    parts = [n for i, n in regions.items() if not i.endswith("R48_BCF")]
    return side_by_side(regions, today_et() - timedelta(weeks=4)) \
        .with_columns((pl.col("Lower 48") - pl.sum_horizontal(parts)).alias("Lower 48 - regions"))


weekly(gas)

The table holds no weekly series yet.


## Refinery utilization and product supplied
Refinery utilization, in percent of operable capacity, and total product supplied, in thousand barrels per day, for the last 8 weeks. Product supplied is the EIA measure of US petroleum demand.

In [10]:
weekly(lambda: side_by_side({"WPULEUS3": "utilization %", "WRPUPUS2": "product supplied"}, weeks))

The table holds no weekly series yet.


## US crude oil production, yearly
The monthly series starts in 1920. The average of each year since 2000, in thousand barrels per day:

In [11]:
crude = to_df(eia("MCRFPUS2", date(2000, 1, 1)), EiaValue)
crude.group_by(pl.col("date").dt.year().alias("year")).agg(pl.col("value").mean().round(0), pl.len().alias("months")).sort("year")

year,value,months
i32,f64,u32
2000,5822.0,12
2001,5801.0,12
2002,5745.0,12
2003,5650.0,12
2004,5441.0,12
…,…,…
2022,12024.0,12
2023,12970.0,12
2024,13267.0,12


## Revisions
The main repo reads the full history of each series on each run. Thus the EIA revisions replace the earlier values, and the table does not keep the earlier values.

## Not here
Other tables hold some EIA data. `fred()` gives the FRED copies: `DCOILWTICO` (daily WTI), `DHHNGSP` (daily Henry Hub) and `GASREGW` (weekly US regular gasoline). `commodities()` gives the futures prices.

## Bad input
An unknown id or name, a filter value that no series has, or a start after the end, raises a clear error before the function reads values.

In [12]:
for call in [lambda: eia("WTI", date(2026, 1, 1)), lambda: eia("DCOILWTICO", date(2026, 1, 1)),
             lambda: eia("RWTC", date(2026, 1, 2), date(2026, 1, 1)),
             lambda: eia_series(frequency="daily"), lambda: eia_series(dataset="eia/natural_gas")]:
    try:
        call()
    except ValueError as e:
        print(e)

'WTI' is not a stored EIA series: use an id or name from eia_series(), such as RWTC or 'WTI crude spot (Cushing)'


'DCOILWTICO' is not a stored EIA series: use an id or name from eia_series(), such as RWTC or 'WTI crude spot (Cushing)'
start must be on or before end


'daily' is not a stored EIA frequency: use one of monthly


'eia/natural_gas' is not a stored EIA dataset: use one of eia/crude_production, eia/natgas, eia/petroleum_spot
